In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

In [4]:
# load training and test datasets
train = pd.read_csv("../Dataset/train.csv")
test  = pd.read_csv("../Dataset/test.csv")

print(f"Train Shape: {train.shape}")
print(f"Test Shape:  {test.shape}")

Train Shape: (630000, 21)
Test Shape:  (270000, 20)


In [5]:
# check column types to identify numeric vs categorical columns
train.dtypes

id                           int64
Soil_Type                   object
Soil_pH                    float64
Soil_Moisture              float64
Organic_Carbon             float64
Electrical_Conductivity    float64
Temperature_C              float64
Humidity                   float64
Rainfall_mm                float64
Sunlight_Hours             float64
Wind_Speed_kmh             float64
Crop_Type                   object
Crop_Growth_Stage           object
Season                      object
Irrigation_Type             object
Water_Source                object
Field_Area_hectare         float64
Mulching_Used               object
Previous_Irrigation_mm     float64
Region                      object
Irrigation_Need             object
dtype: object

In [6]:
# check class balance — critical because we are scored on balanced accuracy
print(train['Irrigation_Need'].value_counts())
print()
print(train['Irrigation_Need'].value_counts(normalize=True))

Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64

Irrigation_Need
Low       0.587170
Medium    0.379483
High      0.033348
Name: proportion, dtype: float64


In [7]:
# check for missing values across all columns
missing = train.isnull().sum().sort_values(ascending=False)
print(missing)

id                         0
Crop_Type                  0
Region                     0
Previous_Irrigation_mm     0
Mulching_Used              0
Field_Area_hectare         0
Water_Source               0
Irrigation_Type            0
Season                     0
Crop_Growth_Stage          0
Wind_Speed_kmh             0
Soil_Type                  0
Sunlight_Hours             0
Rainfall_mm                0
Humidity                   0
Temperature_C              0
Electrical_Conductivity    0
Organic_Carbon             0
Soil_Moisture              0
Soil_pH                    0
Irrigation_Need            0
dtype: int64


In [8]:
# automatically detect categorical columns instead of hardcoding them
# any column with dtype 'object' is categorical
# we exclude the target column 'Irrigation_Need' since we handle that separately
categorical_cols = train.select_dtypes(include='object').columns.tolist()
categorical_cols.remove('Irrigation_Need')

print("Categorical columns found:", categorical_cols)
print()

# show unique values for each categorical column
for col in categorical_cols:
    print(f"{col}: {train[col].nunique()} unique values → {train[col].unique()}")

Categorical columns found: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']

Soil_Type: 4 unique values → ['Loamy' 'Clay' 'Sandy' 'Silt']
Crop_Type: 6 unique values → ['Sugarcane' 'Wheat' 'Rice' 'Potato' 'Cotton' 'Maize']
Crop_Growth_Stage: 4 unique values → ['Sowing' 'Vegetative' 'Flowering' 'Harvest']
Season: 3 unique values → ['Zaid' 'Kharif' 'Rabi']
Irrigation_Type: 4 unique values → ['Drip' 'Rainfed' 'Sprinkler' 'Canal']
Water_Source: 4 unique values → ['Rainwater' 'River' 'Reservoir' 'Groundwater']
Mulching_Used: 2 unique values → ['No' 'Yes']
Region: 5 unique values → ['East' 'South' 'North' 'West' 'Central']


In [9]:
# ── Encode Target ──────────────────────────────────────────────────────────
# keep le_target separate so we can reverse predictions back to words later
le_target = LabelEncoder()
train['Irrigation_Need_encoded'] = le_target.fit_transform(train['Irrigation_Need'])

# print mapping so we never forget which number maps to which class
print("Target mapping:", dict(zip(le_target.classes_, le_target.transform(le_target.classes_))))

# ── Encode Categorical Features ────────────────────────────────────────────
# fit on train only to avoid data leakage
# transform applies the learned mapping without re-learning
le = LabelEncoder()
for col in categorical_cols:
    train[col] = le.fit_transform(train[col])  # learn from train and convert
    test[col]  = le.transform(test[col])        # only convert, never re-learn

print("\nEncoding done!")

Target mapping: {'High': np.int64(0), 'Low': np.int64(1), 'Medium': np.int64(2)}

Encoding done!


In [10]:
# ── Prepare Features and Target ────────────────────────────

X_train = train.drop(['id', 'Irrigation_Need', 'Irrigation_Need_encoded'], axis=1)
X_test  = test.drop(['id'], axis=1)
y_train = train['Irrigation_Need_encoded']

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train shape: {y_train.shape}")

X_train shape: (630000, 19)
X_test shape:  (270000, 19)
y_train shape: (630000,)


In [11]:
# visually confirm columns now contain numbers instead of text
train.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need,Irrigation_Need_encoded
0,0,1,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,2,2,1,1,0.82,0,112.16,1,Low,1
1,1,0,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,3,0,2,3,5.27,1,47.16,3,Low,1
2,2,0,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,3,0,3,2,8.24,1,110.38,2,Low,1
3,3,2,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,0,0,0,3,8.32,1,53.85,3,Medium,2
4,4,0,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,2,1,0,3,7.37,0,93.19,3,Low,1


In [12]:
from sklearn.utils.class_weight import compute_class_weight

# compute weights so model pays more attention to minority class High
classes = np.array([0, 1, 2])
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights = dict(zip(classes, weights))

print("Class weights:", class_weights)

Class weights: {np.int64(0): np.float64(9.995716121662145), np.int64(1): np.float64(0.5676949153458749), np.int64(2): np.float64(0.8783891180136694)}


In [14]:
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score

# 5 fold stratified cross validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_scores = []   # store score from each fold
xgb_models = []   # store model from each fold

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train)):

    # split data into train and validation for this fold
    X_tr  = X_train.iloc[train_idx]
    X_val = X_train.iloc[val_idx]
    y_tr  = y_train.iloc[train_idx]
    y_val = y_train.iloc[val_idx]

    # build model
    model = xgb.XGBClassifier(
        n_estimators=1000,       # number of trees
        learning_rate=0.05,      # how fast the model learns
        max_depth=6,             # how deep each tree grows
        random_state=42,
        eval_metric='mlogloss',
        verbosity=0
    )

    # train model with class weights
    model.fit(
        X_tr, y_tr,
        sample_weight=y_tr.map(class_weights),  # apply class weights
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    # evaluate on validation fold
    preds = model.predict(X_val)
    score = balanced_accuracy_score(y_val, preds)
    xgb_scores.append(score)
    xgb_models.append(model)

    print(f"Fold {fold+1} balanced accuracy: {score:.4f}")

print(f"\nMean XGBoost CV Score: {np.mean(xgb_scores):.4f}")

Fold 1 balanced accuracy: 0.9665
Fold 2 balanced accuracy: 0.9694
Fold 3 balanced accuracy: 0.9696
Fold 4 balanced accuracy: 0.9680
Fold 5 balanced accuracy: 0.9681

Mean XGBoost CV Score: 0.9683


In [15]:
# use the best model (last fold) to predict on test data
# predict returns numbers like 0, 1, 2
raw_predictions = xgb_models[-1].predict(X_test)
print("Raw predictions (numbers):", raw_predictions[:10])

# convert numbers back to words using le_target
final_predictions = le_target.inverse_transform(raw_predictions)
print("Final predictions (words):", final_predictions[:10])

# build submission dataframe
submission = pd.DataFrame({
    'id': test['id'],
    'Irrigation_Need': final_predictions
})

# save to csv
submission.to_csv('submission.csv', index=False)

print(f"\nSubmission saved!")
print(f"Shape: {submission.shape}")
print(f"\nValue counts:\n{submission['Irrigation_Need'].value_counts()}")
print(submission.head(10))

Raw predictions (numbers): [1 1 1 1 1 2 1 2 0 1]
Final predictions (words): ['Low' 'Low' 'Low' 'Low' 'Low' 'Medium' 'Low' 'Medium' 'High' 'Low']

Submission saved!
Shape: (270000, 2)

Value counts:
Irrigation_Need
Low       159811
Medium    101057
High        9132
Name: count, dtype: int64
       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low
5  630005          Medium
6  630006             Low
7  630007          Medium
8  630008            High
9  630009             Low
